# Step 1: Gradient Descent

Bu bölümde Türkçe isimlerle bir bigram modeli oluşturacağım.
Sayım yapmak yerine modelin ağırlıklarını gradient descent eğiteceğim.

## 1.1 Aynı Başlangıç Noktası

Önceki bölümdeki veri kümesini ve karakter tabanlı tokenizer'ı
kullanacağım.

## 1.1 Aynı Başlangıç Noktası

Önceki bölümdeki 1100 Türkçe isimlik veri kümesini ve aynı tokenizer'ı
kullanıyorum. Vocabulary, 29 karakter ve BOS tokenıyla toplam 30 tokendan
oluşuyor.

Model yine mevcut tokenı alıp sonraki tokenların olasılıklarını üretecek.
Değişen kısım, bu olasılıkların hesaplanması ve öğrenme yöntemi olacak.

In [1]:
import random
import math

docs = [
    l.strip().replace(" ", "").replace("İ", "i").replace("I", "ı").lower()
    for l in open("../data/raw_names.txt", encoding="utf-8")
        .read().strip().split("\n")
    if l.strip()
]

random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Vocabulary boyutu:", vocab_size)

İsim sayısı: 1100
Vocabulary boyutu: 30


## 1.2 Parametreler

Modelin parametreleri üç ağırlık matrisinden oluşuyor:

wte: Her tokenı 16 sayılık bir embedding vektörüyle temsil eder.
mlp_fc1: Bu 16 sayıyı 64 sayılık bir ara temsile dönüştürür.
mlp_fc2: Ara temsilden, 30 olası sonraki token için skor üretir.

Parametreler küçük rastgele değerlerle başlatılır.
Eğitim sırasında bu değerler güncellenecektir.

In [2]:
n_embd = 16

matrix = lambda nout, nin: [
    [random.gauss(0, 0.08) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    'wte': matrix(vocab_size, n_embd),
    'mlp_fc1': matrix(4 * n_embd, n_embd),
    'mlp_fc2': matrix(vocab_size, 4 * n_embd),
}

params = [
    (row, j)
    for mat in state_dict.values()
    for row in mat
    for j in range(len(row))
]

In [3]:
for name, mat in state_dict.items():
    print(f"{name}: {len(mat)} × {len(mat[0])}")

print("Toplam parametre:", len(params))

wte: 30 × 16
mlp_fc1: 64 × 16
mlp_fc2: 30 × 64
Toplam parametre: 3424


## 1.3 Linear ve Softmax

Linear fonksiyonu, girdi vektörünü ağırlık matrisiyle çarparak yeni
bir vektör oluşturur. Her çıktı, girdilerin ağırlıklı toplamıdır.

Softmax fonksiyonu, logits adı verilen ham skorları toplamı 1 olan
olasılıklara dönüştürür.

In [4]:
def linear(x, w):
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]


def softmax(logits):
    max_val = max(logits)
    exps = [math.exp(v - max_val) for v in logits]
    total = sum(exps)
    return [e / total for e in exps]

In [5]:
x = [2.0, -1.0, 0.5]

w = [
    [0.3, 0.7, -0.2],
    [-0.5, 0.1, 0.8],
]

print("Linear çıktısı:", linear(x, w))

probs = softmax([2, 1, 0])

print("Softmax çıktısı:", probs)
print("Olasılık toplamı:", sum(probs))

Linear çıktısı: [-0.19999999999999998, -0.7]
Softmax çıktısı: [0.6652409557748218, 0.24472847105479764, 0.09003057317038046]
Olasılık toplamı: 0.9999999999999999


## 1.4 MLP Modeli
MLP (Multi-Layer Perceptron), çok katmanlı algılayıcı anlamına gelir.
Model, mevcut tokenın ID'sini alarak sonraki 30 token için ham skorlar üretir.
ReLU, modele doğrusal olmayan bir işlem kazandırır. Logits değerleri
softmax ile olasılıklara dönüştürülür.

Model hâlâ yalnızca mevcut tokenı kullanır; önceki tokenları hatırlamaz.
Bu aşamada ağırlıklar güncellenmez, yalnızca tahmin hesaplanır.

In [6]:
def mlp(token_id):
    x= state_dict['wte'][token_id]
    x=linear(x,state_dict['mlp_fc1'])
    x=[max(0,xi)for xi in x]#relu
    logits = linear(x,state_dict['mlp_fc2'])
    return logits

In [7]:
token_id = uchars.index("a")

logits = mlp(token_id)
probs = softmax(logits)

print("Skor sayısı:", len(logits))
print("Olasılık sayısı:", len(probs))
print("Olasılıkların toplamı:", sum(probs))

Skor sayısı: 30
Olasılık sayısı: 30
Olasılıkların toplamı: 1.0


## 1.5 Forward Pass

Forward pass, modelin bir isimdeki token geçişleri için tahmin üretmesi
ve ortalama loss değerini hesaplamasıdır.

Her geçişte mevcut token MLP'ye verilir. Üretilen logits değerleri
softmax ile olasılıklara dönüştürülür. Gerçek sonraki tokenın
olasılığından negatif log olasılığı hesaplanır.

İsmin loss değeri, bütün geçişlerin loss değerlerinin ortalamasıdır.
Bu işlem sırasında modelin ağırlıkları değiştirilmez.

In [8]:
def forward(tokens,n):
    losses =[]
    for pos_id in range(n):
        tokens_id, target_id = tokens [pos_id],tokens[pos_id+1]
        logits =mlp(tokens_id)
        probs=softmax(logits)
        loss_t =-math.log(probs[target_id])
        losses.append(loss_t)
    loss =(1/n)* sum(losses)
    return loss

In [9]:
doc = "efe"

tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
n = len(tokens) - 1

loss = forward(tokens, n)

print("İsim:", doc)
print("Tokenlar:", tokens)
print("Geçiş sayısı:", n)
print("Ortalama loss:", loss)

İsim: efe
Tokenlar: [29, 4, 5, 4, 29]
Geçiş sayısı: 4
Ortalama loss: 3.403128193273052


## 1.6 Sayısal Gradyan

Sayısal gradyan, bir parametreyi küçük bir miktar değiştirerek
loss'un nasıl etkilendiğini ölçer.

Önce mevcut loss hesaplanır. Ardından her parametre sırayla küçük
bir miktar artırılır ve loss yeniden hesaplanır. Loss'taki değişim,parametredeki değişime bölünerek yaklaşık türev elde edilir.

Her ölçümden sonra parametre eski değerine döndürülür.
Bu işlem eğitim yapmaz ağırlıkların loss üzerindeki etkisini ölçer.

Her parametre için ayrı bir forward pass gerektiğinden bu yöntem
çok sayıda parametre içeren modellerde yavaştır.

In [10]:
def numerical_gradient(tokens, n):
    loss = forward(tokens, n)
    eps = 1e-5
    grad = []

    for mat in state_dict.values():
        for row in mat:
            for j in range(len(row)):
                old = row[j]
                row[j] = old + eps
                loss_plus = forward(tokens, n)
                row[j] = old
                grad.append((loss_plus - loss) / eps)

    return loss, grad

In [11]:
doc = "efe"
tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
n = len(tokens) - 1

loss, grad = numerical_gradient(tokens, n)

print("İsim:", doc)
print("Loss:", loss)

e_id = uchars.index("e")
start = e_id * n_embd
eps = 1e-5

for j in range(3):
    weight = state_dict['wte'][e_id][j]
    gradient = grad[start + j]

    print(f"\ne harfinin {j + 1}. ağırlığı:")
    print(f"Eski değer: {weight:.8f}")
    print(f"Deneme değeri: {weight + eps:.8f}")
    print(f"Gradyan: {gradient:.8f}")

İsim: efe
Loss: 3.403128193273052

e harfinin 1. ağırlığı:
Eski değer: -0.09458517
Deneme değeri: -0.09457517
Gradyan: 0.01280559

e harfinin 2. ağırlığı:
Eski değer: -0.10947628
Deneme değeri: -0.10946628
Gradyan: 0.00371383

e harfinin 3. ağırlığı:
Eski değer: 0.02035689
Deneme değeri: 0.02036689
Gradyan: -0.00691217


## 1.7 Analitik Gradyan

Bu bölümde modelin parametrelerinin loss üzerindeki etkisini,
ağırlıkları tek tek değiştirip denemek yerine türev kurallarıyla
hesaplayacağız.

İleri geçişte ara değerler saklanır. Geri geçişte zincir kuralı
kullanılarak loss'tan embedding'e doğru gradyanlar hesaplanır.
Bu işleme backpropagation, yani geri yayılım denir.

Gradyanlar her token geçişi için hesaplanıp biriktirilir.
Bu aşamada ağırlıklar güncellenmez.

In [12]:
def analytic_gradient(tokens, n):
    grad = {
        k: [[0.0] * len(row) for row in mat]
        for k, mat in state_dict.items()
    }
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]

        # Forward pass
        x = list(state_dict['wte'][token_id])
        h_pre = linear(x, state_dict['mlp_fc1'])
        h = [max(0, v) for v in h_pre]
        logits = linear(h, state_dict['mlp_fc2'])
        probs = softmax(logits)

        loss_t = -math.log(probs[target_id])
        losses.append(loss_t)

        # Softmax ve loss gradyanı
        dlogits = [p / n for p in probs]
        dlogits[target_id] -= 1.0 / n

        # İkinci linear katman
        dh = [0.0] * len(h)

        for i in range(len(dlogits)):
            for j in range(len(h)):
                grad['mlp_fc2'][i][j] += dlogits[i] * h[j]
                dh[j] += state_dict['mlp_fc2'][i][j] * dlogits[i]

        # ReLU
        dh_pre = [
            dh[j] * (1.0 if h_pre[j] > 0 else 0.0)
            for j in range(len(h_pre))
        ]

        # İlk linear katman
        dx = [0.0] * len(x)

        for i in range(len(dh_pre)):
            for j in range(len(x)):
                grad['mlp_fc1'][i][j] += dh_pre[i] * x[j]
                dx[j] += state_dict['mlp_fc1'][i][j] * dh_pre[i]

        # Embedding
        for j in range(len(x)):
            grad['wte'][token_id][j] += dx[j]

    loss = (1 / n) * sum(losses)
    grad_flat = [g for mat in grad.values() for row in mat for g in row]
    return loss, grad_flat

In [13]:
doc = "efe"
tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
n = len(tokens) - 1

loss, grad = analytic_gradient(tokens, n)

print("Loss:", loss)
print("Parametre sayısı:", len(params))
print("Gradyan sayısı:", len(grad))

e_id = uchars.index("e")
start = e_id * n_embd

print("e embedding gradyanları:", grad[start:start + n_embd])

Loss: 3.403128193273052
Parametre sayısı: 3424
Gradyan sayısı: 3424
e embedding gradyanları: [0.012805587396344884, 0.003713823172713961, -0.006912176305124138, -0.0013019317219858909, -0.014724711444237527, -0.016960513719825845, -4.162274148288318e-06, 0.012431248027456735, -0.014054247014649328, -0.014440116402281293, 0.014755079915208667, 0.009256331376811533, -0.0038925583942803817, -0.017676326183993497, -0.019844240655999504, 0.0050974532929908565]
